# SegFormer MiT-B2 Image Forgery Segmentation - GCS 1,000 Sample

This notebook follows the same overall flow as the Kaggle SegFormer MiT-B2 image-forgery segmentation notebook: install packages, load data, prepare image/mask pairs, build PyTorch datasets, define SegFormer, train, validate, test, and visualize predictions.

Dataset source:

`gs://maverick91/audit_finding_project/screenshot_tamper_final/`

Experiment requested here:

- Randomly sample **1,000** base screenshot IDs from GCS manifests.
- Select one tampered image + mask pair per base screenshot.
- Record every sampled `image_id`, `tamper_id`, split, alteration, image URI, and mask URI.
- Create exact **70/15/15** train/validation/test splits: 700 / 150 / 150.
- Train SegFormer MiT-B2 for up to **50 epochs**.
- Run **three learning-rate trials** and choose the best validation Dice result.
- Stop each trial when validation Dice does not improve for **3 epochs**.
- Return the best trial, learning rate, and best epoch, then evaluate that checkpoint on the test split.

Outputs written locally:

- `outputs/sampled_1000_splits.csv`
- `outputs/trial_history.csv`
- `outputs/trial_summary.csv`
- `outputs/best_trial_summary.json`
- `outputs/test_metrics.json`
- `outputs/checkpoints/.../best.pt`
- `outputs/best_model_test_predictions.png`

# Install Required Segmentation Packages

Run this once at the start of a Colab/Kaggle session. If your runtime already has the packages, this will be quick.

In [ ]:
%pip install -q segmentation-models-pytorch albumentations opencv-python-headless google-cloud-storage pandas scikit-learn matplotlib tqdm

# Data Loading and Basic Statistics - Imports and Experiment Configuration

Update `LEARNING_RATE_TRIALS`, `BATCH_SIZE`, or `IMAGE_SIZE` if your GPU memory requires it. The default three LR trials are intentionally conservative for fine-tuning a pretrained transformer backbone on a small subset.

In [ ]:
import json
import math
import os
import random
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import albumentations as A
from albumentations.pytorch import ToTensorV2

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

import segmentation_models_pytorch as smp
from google.cloud import storage

# -------------------------
# Reproducibility / sampling
# -------------------------
SEED = 42
SAMPLE_SIZE = 1000
SPLIT_RATIOS = (0.70, 0.15, 0.15)
EXPECTED_SPLIT_COUNTS = {"train": 700, "val": 150, "test": 150}

# Sample one tampered image per base screenshot manifest. This gives exactly
# 1,000 image/mask training examples and avoids multiple alterations from the
# same base screenshot leaking across splits.
SAMPLE_MODE = "one_tamper_per_base_image"

# -------------------------
# GCS source
# -------------------------
GCS_BUCKET = "maverick91"
GCS_PREFIX = "audit_finding_project/screenshot_tamper_final"
MANIFEST_PREFIX = f"{GCS_PREFIX}/manifests"
IMAGE_PREFIX = f"{GCS_PREFIX}/images"

# -------------------------
# Training config
# -------------------------
ENCODER_NAME = "mit_b2"
ENCODER_WEIGHTS = "imagenet"
IMAGE_SIZE = 512
BATCH_SIZE = 4
NUM_WORKERS = 2
MAX_EPOCHS = 50
EARLY_STOP_PATIENCE = 3  # tolerance of 3 non-improving validation epochs
MIN_DELTA = 1e-4

# The first run selected 1e-4 as best, so keep three trials but search near it.
LEARNING_RATE_TRIALS = [3e-5, 1e-4, 2e-4]
WEIGHT_DECAY = 1e-5
MIXED_PRECISION = True
GRADIENT_CLIP_NORM = 1.0

# Tamper masks are tiny relative to the full screenshot; these settings reduce
# the background-only bias that makes pixel accuracy look high while Dice is low.
LOSS_NAME = "weighted_bce_focal_tversky"
FOCAL_TVERSKY_ALPHA = 0.3  # false-positive weight
FOCAL_TVERSKY_BETA = 0.7   # false-negative weight; higher encourages recall
FOCAL_TVERSKY_GAMMA = 0.75
BCE_WEIGHT = 0.35
TVERSKY_WEIGHT = 0.65
POS_WEIGHT_CAP = 30.0
BALANCE_TRAIN_BY_ALTERATION = True
THRESHOLD = 0.35
THRESHOLD_SWEEP = [float(round(x, 2)) for x in np.arange(0.10, 0.71, 0.05)]

# -------------------------
# Local output/cache paths
# -------------------------
try:
    from google.colab import drive  # noqa: F401
    DEFAULT_ROOT = Path("/content/segformer_gcs_1000")
except Exception:
    DEFAULT_ROOT = Path.cwd() / "segformer_gcs_1000"

RUN_NAME = datetime.utcnow().strftime("segformer_mit_b2_1000_%Y%m%d_%H%M%S")
WORK_DIR = DEFAULT_ROOT
CACHE_DIR = WORK_DIR / "cache"
OUTPUT_DIR = WORK_DIR / "outputs" / RUN_NAME
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"

for path in [WORK_DIR, CACHE_DIR, OUTPUT_DIR, CHECKPOINT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

# Optional: set to True if your service account has write access and you want
# the notebook outputs uploaded back to the bucket.
UPLOAD_OUTPUTS_TO_GCS = False
OUTPUT_GCS_PREFIX = f"{GCS_PREFIX}/segformer_results/{RUN_NAME}"

print(f"Run name: {RUN_NAME}")
print(f"Work dir: {WORK_DIR}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# GCS Service Account Setup

This cell does **not** ask for an upload. It reads credentials in this order:

1. `GOOGLE_APPLICATION_CREDENTIALS`, if already set.
2. A previously-created key file at `/content/gcs_sa.json`, `/kaggle/working/gcs_sa.json`, or the notebook work dir.
3. A `GCS_SA_KEY` Python variable, if you pasted the service-account JSON string into this cell or a cell above it.
4. A Colab Secret named `GCS_SA_JSON`.
5. Application Default Credentials/public bucket access.

If you want to use the teammate key directly, paste it into `GCS_SA_KEY = r'''...'''` in the code cell below. Do not commit or share a notebook containing a real private key.

In [ ]:
# Paste the service-account JSON string here only in your private runtime if you
# do not want to use Colab Secrets or GOOGLE_APPLICATION_CREDENTIALS.
# Example:
# GCS_SA_KEY = r'''{"type":"service_account", ... }'''
GCS_SA_KEY = globals().get("GCS_SA_KEY", "").strip()


def _write_gcs_key_json(raw_key: str, key_path: Path) -> bool:
    if not raw_key:
        return False
    try:
        parsed = json.loads(raw_key)
    except json.JSONDecodeError as exc:
        raise ValueError("GCS_SA_KEY/GCS_SA_JSON is not valid JSON") from exc
    if parsed.get("type") != "service_account" or not parsed.get("client_email"):
        raise ValueError("Credential JSON does not look like a service-account key")

    # Fix common copy/paste issue: private_key sometimes arrives with literal
    # backslash-n sequences, which breaks PEM loading in google-auth.
    if parsed.get("private_key"):
        parsed["private_key"] = parsed["private_key"].replace("\\n", "\n")

    key_path.write_text(json.dumps(parsed))
    key_path.chmod(0o600)
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = str(key_path)
    print(f"Wrote service-account key to {key_path}")
    return True


def configure_gcs_credentials(work_dir: Path) -> None:
    """Configure credentials without an interactive upload prompt."""
    existing = os.environ.get("GOOGLE_APPLICATION_CREDENTIALS")
    if existing and Path(existing).exists():
        print(f"Using GOOGLE_APPLICATION_CREDENTIALS={existing}")
        return

    candidate_paths = [
        work_dir / "gcs_sa.json",
        Path("/content/gcs_sa.json"),
        Path("/kaggle/working/gcs_sa.json"),
    ]
    for candidate in candidate_paths:
        if candidate.exists():
            os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = str(candidate)
            print(f"Using service-account file: {candidate}")
            return

    key_path = work_dir / "gcs_sa.json"
    if _write_gcs_key_json(GCS_SA_KEY, key_path):
        return

    # Colab Secrets path: store the whole JSON string in a secret named GCS_SA_JSON.
    try:
        from google.colab import userdata
        secret_json = userdata.get("GCS_SA_JSON")
        if _write_gcs_key_json((secret_json or "").strip(), key_path):
            return
    except Exception as exc:
        print(f"Colab secret GCS_SA_JSON not available: {type(exc).__name__}")

    print("No explicit key configured. Trying Application Default Credentials/public access.")


configure_gcs_credentials(WORK_DIR)
client = storage.Client()
bucket = client.bucket(GCS_BUCKET)

# Smoke-test access.
probe = next(client.list_blobs(GCS_BUCKET, prefix=MANIFEST_PREFIX + "/", max_results=1), None)
if probe is None:
    raise RuntimeError(f"No manifests found under gs://{GCS_BUCKET}/{MANIFEST_PREFIX}/")
print(f"GCS access OK. First manifest-like object: gs://{GCS_BUCKET}/{probe.name}")

# Data Loader Helper Function

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True


def gcs_uri(blob_name: str) -> str:
    return f"gs://{GCS_BUCKET}/{blob_name}"


def safe_local_name(blob_name: str) -> str:
    return blob_name.replace("/", "__")


def list_manifest_blobs() -> list[str]:
    blobs = []
    prefix = MANIFEST_PREFIX.rstrip("/") + "/"
    for blob in tqdm(client.list_blobs(GCS_BUCKET, prefix=prefix), desc="Listing manifests"):
        if blob.name.endswith("_manifest.json"):
            blobs.append(blob.name)
    blobs = sorted(blobs)
    if not blobs:
        raise RuntimeError(f"No manifest JSON files found under gs://{GCS_BUCKET}/{prefix}")
    return blobs


def load_manifest(blob_name: str) -> dict:
    blob = bucket.blob(blob_name)
    return json.loads(blob.download_as_text())


def make_row_from_manifest(manifest_blob: str, manifest: dict, rng: random.Random) -> dict | None:
    entries = manifest.get("generated", []) or []
    valid_entries = []
    for entry in entries:
        files = entry.get("files", {}) or {}
        if files.get("tampered") and files.get("mask"):
            valid_entries.append(entry)
    if not valid_entries:
        return None

    entry = rng.choice(valid_entries)
    files = entry["files"]
    image_id = manifest.get("image_id") or Path(manifest_blob).name.replace("_manifest.json", "")
    tamper_id = entry.get("tamper_id") or Path(files["tampered"]).stem.replace("_tampered", "")

    image_blob = f"{IMAGE_PREFIX}/{files['tampered']}"
    mask_blob = f"{IMAGE_PREFIX}/{files['mask']}"
    clean_blob = f"{IMAGE_PREFIX}/{manifest.get('clean_file')}" if manifest.get("clean_file") else None

    return {
        "image_id": image_id,
        "tamper_id": tamper_id,
        "alteration": entry.get("alteration"),
        "dom_index": entry.get("dom_index"),
        "dom_type": entry.get("dom_type"),
        "bbox": json.dumps(entry.get("bbox") or entry.get("dst_bbox") or []),
        "manifest_blob": manifest_blob,
        "clean_blob": clean_blob,
        "image_blob": image_blob,
        "mask_blob": mask_blob,
        "image_gcs_uri": gcs_uri(image_blob),
        "mask_gcs_uri": gcs_uri(mask_blob),
    }


def download_blob(blob_name: str, destination: Path) -> Path:
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() and destination.stat().st_size > 0:
        return destination
    bucket.blob(blob_name).download_to_filename(str(destination))
    return destination


def download_dataset_files(df: pd.DataFrame, cache_dir: Path, max_workers: int = 16) -> pd.DataFrame:
    image_dir = cache_dir / "images"
    mask_dir = cache_dir / "masks"
    image_dir.mkdir(parents=True, exist_ok=True)
    mask_dir.mkdir(parents=True, exist_ok=True)

    jobs = []
    df = df.copy()
    df["local_image_path"] = [str(image_dir / safe_local_name(x)) for x in df["image_blob"]]
    df["local_mask_path"] = [str(mask_dir / safe_local_name(x)) for x in df["mask_blob"]]

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        for row in df.itertuples(index=False):
            jobs.append(executor.submit(download_blob, row.image_blob, Path(row.local_image_path)))
            jobs.append(executor.submit(download_blob, row.mask_blob, Path(row.local_mask_path)))
        for future in tqdm(as_completed(jobs), total=len(jobs), desc="Downloading image/mask files"):
            future.result()
    return df


def upload_file_to_gcs(local_path: Path, remote_blob_name: str) -> None:
    bucket.blob(remote_blob_name).upload_from_filename(str(local_path))


def upload_outputs_to_gcs(output_dir: Path, output_prefix: str) -> None:
    files = [path for path in output_dir.rglob("*") if path.is_file()]
    for path in tqdm(files, desc="Uploading outputs to GCS"):
        relative = path.relative_to(output_dir).as_posix()
        upload_file_to_gcs(path, f"{output_prefix.rstrip('/')}/{relative}")
    print(f"Uploaded {len(files)} files to gs://{GCS_BUCKET}/{output_prefix}/")

set_seed(SEED)

# Data Preparation

Sample 1,000 random base image IDs, pick one tampered image/mask pair per base image, and create the requested 70/15/15 split. The split CSV records both the base `image_id` and selected `tamper_id`, plus GCS URIs for the tampered image and mask.

In [ ]:
rng = random.Random(SEED)
manifest_blobs = list_manifest_blobs()
print(f"Found {len(manifest_blobs):,} manifests")

if len(manifest_blobs) < SAMPLE_SIZE:
    raise ValueError(f"Requested {SAMPLE_SIZE} samples but only found {len(manifest_blobs)} manifests")


def row_files_exist(row: dict) -> bool:
    return (
        bucket.blob(row["image_blob"]).exists(client)
        and bucket.blob(row["mask_blob"]).exists(client)
    )


candidate_manifest_blobs = manifest_blobs.copy()
rng.shuffle(candidate_manifest_blobs)
rows = []
failed_manifests = []
missing_file_rows = []
BATCH_MANIFESTS = 2048

# Walk the shuffled manifest list until we have exactly 1,000 valid examples.
# This avoids ending up with fewer examples after missing/invalid manifests.
for start in tqdm(range(0, len(candidate_manifest_blobs), BATCH_MANIFESTS), desc="Sampling valid manifests"):
    if len(rows) >= SAMPLE_SIZE:
        break
    batch_blobs = candidate_manifest_blobs[start : start + BATCH_MANIFESTS]
    loaded = {}
    with ThreadPoolExecutor(max_workers=32) as executor:
        future_to_blob = {executor.submit(load_manifest, blob): blob for blob in batch_blobs}
        for future in as_completed(future_to_blob):
            blob = future_to_blob[future]
            try:
                loaded[blob] = future.result()
            except Exception as exc:
                failed_manifests.append(f"{blob}\t{type(exc).__name__}: {exc}")

    for blob in batch_blobs:
        if len(rows) >= SAMPLE_SIZE:
            break
        manifest = loaded.get(blob)
        if manifest is None:
            continue
        row = make_row_from_manifest(blob, manifest, rng)
        if row is None:
            failed_manifests.append(f"{blob}\tno valid generated tamper entries")
            continue
        if not row_files_exist(row):
            missing_file_rows.append(row)
            continue
        rows.append(row)

if len(rows) != SAMPLE_SIZE:
    raise RuntimeError(f"Could only build {len(rows)} valid rows out of requested {SAMPLE_SIZE}")

sample_df = pd.DataFrame(rows)
sample_df = sample_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

n_train = EXPECTED_SPLIT_COUNTS["train"]
n_val = EXPECTED_SPLIT_COUNTS["val"]
n_test = EXPECTED_SPLIT_COUNTS["test"]
assert n_train + n_val + n_test == SAMPLE_SIZE, "Expected split counts must sum to SAMPLE_SIZE"

sample_df.loc[: n_train - 1, "split"] = "train"
sample_df.loc[n_train : n_train + n_val - 1, "split"] = "val"
sample_df.loc[n_train + n_val :, "split"] = "test"

split_counts = sample_df["split"].value_counts().to_dict()
if split_counts != EXPECTED_SPLIT_COUNTS:
    raise AssertionError(f"Split count mismatch: got {split_counts}, expected {EXPECTED_SPLIT_COUNTS}")
if sample_df["image_id"].nunique() != SAMPLE_SIZE:
    raise AssertionError("Expected one unique base image_id per sampled row")
if sample_df["tamper_id"].nunique() != SAMPLE_SIZE:
    raise AssertionError("Expected one unique tamper_id per sampled row")

print(f"Split counts: {split_counts}")
print(sample_df[["split", "image_id", "tamper_id", "alteration", "image_gcs_uri", "mask_gcs_uri"]].head())

split_csv = OUTPUT_DIR / "sampled_1000_splits.csv"
sample_df.to_csv(split_csv, index=False)
print(f"Wrote split/ID record to {split_csv}")

if failed_manifests:
    failed_path = OUTPUT_DIR / "failed_selected_manifests.txt"
    failed_path.write_text("\n".join(failed_manifests))
    print(f"Skipped {len(failed_manifests)} invalid manifests; wrote details to {failed_path}")
if missing_file_rows:
    missing_path = OUTPUT_DIR / "missing_sample_files.csv"
    pd.DataFrame(missing_file_rows).to_csv(missing_path, index=False)
    print(f"Skipped {len(missing_file_rows)} rows with missing files; wrote details to {missing_path}")

# Data Visualization

Inspect the sampled train/validation/test counts and alteration distribution before downloading/training.

In [ ]:
print("=" * 60)
print(" SAMPLED DATA STATISTICS")
print("=" * 60)
print(f"Total sampled examples: {len(sample_df):,}")
print(f"Unique base image IDs: {sample_df['image_id'].nunique():,}")
print(f"Unique tamper IDs: {sample_df['tamper_id'].nunique():,}")
print("\nSplit counts:")
print(sample_df['split'].value_counts().reindex(['train', 'val', 'test']))
print("\nAlteration counts:")
print(sample_df['alteration'].value_counts(dropna=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sample_df['split'].value_counts().reindex(['train', 'val', 'test']).plot(kind='bar', ax=axes[0], color=['#4C72B0', '#55A868', '#C44E52'])
axes[0].set_title('70/15/15 split counts')
axes[0].set_xlabel('split')
axes[0].set_ylabel('examples')
sample_df['alteration'].value_counts(dropna=False).plot(kind='bar', ax=axes[1], color='#8172B2')
axes[1].set_title('sampled alteration types')
axes[1].set_xlabel('alteration')
axes[1].set_ylabel('examples')
fig.tight_layout()
summary_plot_path = OUTPUT_DIR / 'sampled_data_summary.png'
fig.savefig(summary_plot_path, dpi=160, bbox_inches='tight')
plt.show()
print(f"Wrote {summary_plot_path}")

# Download Sampled Images and Masks

Training reads local files for speed and to avoid GCS client/thread contention inside PyTorch workers.

In [ ]:
sample_df = download_dataset_files(sample_df, CACHE_DIR, max_workers=24)

if len(sample_df) != SAMPLE_SIZE:
    raise AssertionError(f"Cached row count mismatch: got {len(sample_df)}, expected {SAMPLE_SIZE}")
post_cache_counts = sample_df["split"].value_counts().to_dict()
if post_cache_counts != EXPECTED_SPLIT_COUNTS:
    raise AssertionError(f"Post-cache split count mismatch: got {post_cache_counts}, expected {EXPECTED_SPLIT_COUNTS}")

mask_positive_pixels = []
mask_total_pixels = []
for mask_path in tqdm(sample_df["local_mask_path"], desc="Checking all masks"):
    arr = np.array(Image.open(mask_path).convert("L"))
    positive = int((arr > 0).sum())
    total = int(arr.size)
    mask_positive_pixels.append(positive)
    mask_total_pixels.append(total)

sample_df["mask_positive_pixels"] = mask_positive_pixels
sample_df["mask_total_pixels"] = mask_total_pixels
sample_df["mask_positive_ratio"] = sample_df["mask_positive_pixels"] / sample_df["mask_total_pixels"].clip(lower=1)
empty_masks = int((sample_df["mask_positive_pixels"] == 0).sum())
if empty_masks:
    empty_path = OUTPUT_DIR / "empty_masks.csv"
    sample_df[sample_df["mask_positive_pixels"] == 0].to_csv(empty_path, index=False)
    raise AssertionError(f"Found {empty_masks} empty masks; wrote {empty_path}")

indexed_csv = OUTPUT_DIR / "sampled_1000_splits_with_local_paths.csv"
sample_df.to_csv(indexed_csv, index=False)
print(f"Cached {len(sample_df)} examples and wrote {indexed_csv}")
print(f"Masks with tampered pixels: {(sample_df['mask_positive_pixels'] > 0).sum()}/{len(sample_df)}")
print(f"Median positive mask ratio: {sample_df['mask_positive_ratio'].median():.6f}")
print(f"Mean positive mask ratio: {sample_df['mask_positive_ratio'].mean():.6f}")

# Data Augmentation and PyTorch Dataset Generator

Define Albumentations transforms, the PyTorch Dataset, train/validation/test DataLoaders, SegFormer wrapper, losses, and segmentation metrics.

In [ ]:
class TamperSegmentationDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        image = np.array(Image.open(row.local_image_path).convert("RGB"))
        mask = np.array(Image.open(row.local_mask_path).convert("L"))
        mask = (mask > 0).astype("float32")

        if self.transform:
            transformed = self.transform(image=image, mask=mask)
            image = transformed["image"]
            mask = transformed["mask"]
        else:
            image = torch.from_numpy(image.transpose(2, 0, 1)).float() / 255.0
            mask = torch.from_numpy(mask).float()

        if mask.ndim == 2:
            mask = mask.unsqueeze(0)
        else:
            mask = mask.permute(2, 0, 1)

        return {
            "image": image.float(),
            "mask": mask.float(),
            "image_id": row.image_id,
            "tamper_id": row.tamper_id,
            "alteration": row.alteration,
        }


def build_transforms(image_size: int):
    train_transform = A.Compose([
        A.Resize(image_size, image_size, interpolation=cv2.INTER_LINEAR, mask_interpolation=cv2.INTER_NEAREST),
        A.HorizontalFlip(p=0.5),
        A.Affine(scale=(0.90, 1.10), translate_percent=(-0.03, 0.03), rotate=(-3, 3), p=0.35),
        A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.30),
        A.GaussNoise(p=0.15),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])
    eval_transform = A.Compose([
        A.Resize(image_size, image_size, interpolation=cv2.INTER_LINEAR, mask_interpolation=cv2.INTER_NEAREST),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])
    return train_transform, eval_transform


class SegformerForgeryDetector(nn.Module):
    def __init__(self, encoder_name=ENCODER_NAME, encoder_weights=ENCODER_WEIGHTS):
        super().__init__()
        self.model = smp.Segformer(
            encoder_name=encoder_name,
            encoder_weights=encoder_weights,
            in_channels=3,
            classes=1,
            activation=None,
        )

    def forward(self, x):
        return self.model(x)


class SoftDiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        dims = (1, 2, 3)
        intersection = (probs * targets).sum(dim=dims)
        denominator = probs.sum(dim=dims) + targets.sum(dim=dims)
        dice = (2.0 * intersection + self.smooth) / (denominator + self.smooth)
        return 1.0 - dice.mean()


class FocalTverskyLoss(nn.Module):
    def __init__(self, alpha=0.3, beta=0.7, gamma=0.75, smooth=1.0):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        dims = (1, 2, 3)
        tp = (probs * targets).sum(dim=dims)
        fp = (probs * (1.0 - targets)).sum(dim=dims)
        fn = ((1.0 - probs) * targets).sum(dim=dims)
        tversky = (tp + self.smooth) / (tp + self.alpha * fp + self.beta * fn + self.smooth)
        return torch.pow(1.0 - tversky, self.gamma).mean()


class CombinedLoss(nn.Module):
    def __init__(self, pos_weight=None, bce_weight=BCE_WEIGHT, tversky_weight=TVERSKY_WEIGHT):
        super().__init__()
        self.bce_weight = bce_weight
        self.tversky_weight = tversky_weight
        self.bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        self.tversky = FocalTverskyLoss(
            alpha=FOCAL_TVERSKY_ALPHA,
            beta=FOCAL_TVERSKY_BETA,
            gamma=FOCAL_TVERSKY_GAMMA,
        )
        self.soft_dice = SoftDiceLoss()

    def forward(self, logits, targets):
        bce_loss = self.bce(logits, targets)
        tversky_loss = self.tversky(logits, targets)
        # Small Dice term stabilizes overlap optimization while Tversky focuses recall.
        overlap_loss = 0.8 * tversky_loss + 0.2 * self.soft_dice(logits, targets)
        total = self.bce_weight * bce_loss + self.tversky_weight * overlap_loss
        return total, bce_loss, overlap_loss


class AverageMeter:
    def __init__(self):
        self.reset()

    def reset(self):
        self.sum = 0.0
        self.count = 0
        self.avg = 0.0

    def update(self, value, n=1):
        self.sum += float(value) * n
        self.count += n
        self.avg = self.sum / max(self.count, 1)


def batch_metrics(logits, targets, threshold=THRESHOLD, smooth=1.0):
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()

    dims = (1, 2, 3)
    intersection = (preds * targets).sum(dim=dims)
    pred_sum = preds.sum(dim=dims)
    target_sum = targets.sum(dim=dims)
    union = pred_sum + target_sum - intersection

    dice = ((2.0 * intersection + smooth) / (pred_sum + target_sum + smooth)).mean().item()
    iou = ((intersection + smooth) / (union + smooth)).mean().item()
    pixel_acc = (preds == targets).float().mean().item()
    return dice, iou, pixel_acc


class EarlyStopper:
    def __init__(self, patience=3, min_delta=1e-4, mode="max"):
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.best = None
        self.bad_epochs = 0

    def improved(self, value):
        if self.best is None:
            return True
        if self.mode == "max":
            return value > self.best + self.min_delta
        return value < self.best - self.min_delta

    def step(self, value):
        if self.improved(value):
            self.best = value
            self.bad_epochs = 0
            return False, True
        self.bad_epochs += 1
        return self.bad_epochs >= self.patience, False


train_transform, eval_transform = build_transforms(IMAGE_SIZE)
train_df = sample_df[sample_df.split == "train"].reset_index(drop=True)
val_df = sample_df[sample_df.split == "val"].reset_index(drop=True)
test_df = sample_df[sample_df.split == "test"].reset_index(drop=True)

actual_split_counts = {"train": len(train_df), "val": len(val_df), "test": len(test_df)}
if actual_split_counts != EXPECTED_SPLIT_COUNTS:
    raise AssertionError(f"DataFrame split mismatch: got {actual_split_counts}, expected {EXPECTED_SPLIT_COUNTS}")

train_positive = float(train_df["mask_positive_pixels"].sum())
train_total = float(train_df["mask_total_pixels"].sum())
train_negative = max(train_total - train_positive, 1.0)
POS_WEIGHT_VALUE = min(train_negative / max(train_positive, 1.0), POS_WEIGHT_CAP)
POS_WEIGHT_TENSOR = torch.tensor([POS_WEIGHT_VALUE], dtype=torch.float32)
print(f"Train foreground ratio: {train_positive / train_total:.6f}")
print(f"BCE positive class weight: {POS_WEIGHT_VALUE:.2f} (cap={POS_WEIGHT_CAP})")


def build_train_sampler(dataframe: pd.DataFrame):
    if not BALANCE_TRAIN_BY_ALTERATION:
        return None
    alteration = dataframe["alteration"].fillna("unknown")
    inverse_freq = alteration.map(1.0 / alteration.value_counts()).astype(float)
    mask_ratio = dataframe["mask_positive_ratio"].astype(float)
    median_ratio = max(float(mask_ratio.median()), 1e-8)
    area_boost = np.clip(mask_ratio / median_ratio, 0.5, 2.0)
    weights = inverse_freq.to_numpy() * area_boost.to_numpy()
    weights = weights / weights.mean()
    return WeightedRandomSampler(
        weights=torch.as_tensor(weights, dtype=torch.double),
        num_samples=len(dataframe),
        replacement=True,
    )


train_sampler = build_train_sampler(train_df)
train_loader = DataLoader(
    TamperSegmentationDataset(train_df, train_transform),
    batch_size=BATCH_SIZE,
    shuffle=train_sampler is None,
    sampler=train_sampler,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    TamperSegmentationDataset(val_df, eval_transform),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
test_loader = DataLoader(
    TamperSegmentationDataset(test_df, eval_transform),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
POS_WEIGHT_TENSOR = POS_WEIGHT_TENSOR.to(device)
print(f"Train/val/test: {len(train_df)}/{len(val_df)}/{len(test_df)}")
print(f"Balanced train sampler: {train_sampler is not None}")
print(f"Device: {device}")

# Validate DataLoader

Check one training batch before running model trials.

In [ ]:
sample_batch = next(iter(train_loader))
print("=" * 60)
print(" VALIDATING DATALOADER")
print("=" * 60)
print(f"Image batch shape: {sample_batch['image'].shape}")
print(f"Mask batch shape : {sample_batch['mask'].shape}")
print(f"Image dtype      : {sample_batch['image'].dtype}")
print(f"Mask dtype       : {sample_batch['mask'].dtype}")
print(f"First image_id   : {sample_batch['image_id'][0]}")
print(f"First tamper_id  : {sample_batch['tamper_id'][0]}")
print(f"First alteration : {sample_batch['alteration'][0]}")
print(f"Mask positive pixels in first batch: {sample_batch['mask'].sum(dim=(1, 2, 3)).tolist()}")

# Model Architecture

Create SegFormer MiT-B2 with ImageNet encoder weights, print parameter counts, and run a small forward-pass check.

In [ ]:
print("=" * 60)
print(" MODEL ARCHITECTURE")
print("=" * 60)

architecture_model = SegformerForgeryDetector().to(device)

def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

total_params, trainable_params = count_parameters(architecture_model)
print("SegFormer model created")
print(f"  - Encoder: {ENCODER_NAME}")
print(f"  - Pretrained weights: {ENCODER_WEIGHTS}")
print("  - Input channels: 3 RGB")
print("  - Output classes: 1 binary mask")
print(f"  - Total parameters: {total_params:,}")
print(f"  - Trainable parameters: {trainable_params:,}")
print(f"  - Device: {device}")

with torch.no_grad():
    dummy_input = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE, device=device)
    dummy_output = architecture_model(dummy_input)
print(f"Forward pass input shape : {tuple(dummy_input.shape)}")
print(f"Forward pass output shape: {tuple(dummy_output.shape)}")

del architecture_model, dummy_input, dummy_output
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Configuration

Confirm the requested training settings before launching the three learning-rate trials.

In [ ]:
CONFIG = {
    'encoder_name': ENCODER_NAME,
    'encoder_weights': ENCODER_WEIGHTS,
    'image_size': IMAGE_SIZE,
    'batch_size': BATCH_SIZE,
    'epochs': MAX_EPOCHS,
    'learning_rate_trials': LEARNING_RATE_TRIALS,
    'early_stopping_patience': EARLY_STOP_PATIENCE,
    'early_stopping_metric': 'val_dice',
    'min_delta': MIN_DELTA,
    'weight_decay': WEIGHT_DECAY,
    'mixed_precision': MIXED_PRECISION,
    'gradient_clip_norm': GRADIENT_CLIP_NORM,
    'loss_name': LOSS_NAME,
    'bce_weight': BCE_WEIGHT,
    'tversky_weight': TVERSKY_WEIGHT,
    'focal_tversky_alpha': FOCAL_TVERSKY_ALPHA,
    'focal_tversky_beta': FOCAL_TVERSKY_BETA,
    'focal_tversky_gamma': FOCAL_TVERSKY_GAMMA,
    'pos_weight_value': POS_WEIGHT_VALUE,
    'balance_train_by_alteration': BALANCE_TRAIN_BY_ALTERATION,
    'threshold_sweep': THRESHOLD_SWEEP,
    'split_counts': sample_df['split'].value_counts().reindex(['train', 'val', 'test']).to_dict(),
    'checkpoint_dir': str(CHECKPOINT_DIR),
}

print("=" * 60)
print(" TRAINING CONFIGURATION")
print("=" * 60)
print(json.dumps(CONFIG, indent=2))

# Training and Evaluation Helper Functions

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None, scaler=None, epoch=0, phase="train"):
    is_train = optimizer is not None
    model.train(is_train)

    loss_meter = AverageMeter()
    bce_meter = AverageMeter()
    dice_loss_meter = AverageMeter()
    dice_meter = AverageMeter()
    iou_meter = AverageMeter()
    acc_meter = AverageMeter()

    pbar = tqdm(loader, desc=f"{phase.title()} epoch {epoch}", leave=False)
    for batch in pbar:
        images = batch["image"].to(device, non_blocking=True)
        masks = batch["mask"].to(device, non_blocking=True)
        batch_size = images.size(0)

        with torch.set_grad_enabled(is_train):
            with torch.cuda.amp.autocast(enabled=MIXED_PRECISION and device.type == "cuda"):
                logits = model(images)
                loss, bce_loss, dice_loss = criterion(logits, masks)

            if is_train:
                optimizer.zero_grad(set_to_none=True)
                if scaler is not None and scaler.is_enabled():
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP_NORM)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP_NORM)
                    optimizer.step()

        with torch.no_grad():
            dice, iou, acc = batch_metrics(logits.detach(), masks.detach())

        loss_meter.update(loss.item(), batch_size)
        bce_meter.update(bce_loss.item(), batch_size)
        dice_loss_meter.update(dice_loss.item(), batch_size)
        dice_meter.update(dice, batch_size)
        iou_meter.update(iou, batch_size)
        acc_meter.update(acc, batch_size)
        pbar.set_postfix(loss=f"{loss_meter.avg:.4f}", dice=f"{dice_meter.avg:.4f}", iou=f"{iou_meter.avg:.4f}")

    return {
        "loss": loss_meter.avg,
        "bce_loss": bce_meter.avg,
        "dice_loss": dice_loss_meter.avg,
        "dice": dice_meter.avg,
        "iou": iou_meter.avg,
        "pixel_acc": acc_meter.avg,
    }


def save_checkpoint(path: Path, model, optimizer, epoch: int, lr: float, metrics: dict, trial_index: int) -> None:
    checkpoint = {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "epoch": epoch,
        "learning_rate": lr,
        "trial_index": trial_index,
        "metrics": metrics,
        "encoder_name": ENCODER_NAME,
        "encoder_weights": ENCODER_WEIGHTS,
        "image_size": IMAGE_SIZE,
        "threshold": THRESHOLD,
        "sample_ids_csv": str(split_csv),
    }
    torch.save(checkpoint, path)


def train_trial(trial_index: int, learning_rate: float) -> tuple[dict, list[dict]]:
    set_seed(SEED + trial_index)
    model = SegformerForgeryDetector().to(device)
    criterion = CombinedLoss(pos_weight=POS_WEIGHT_TENSOR)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=2, min_lr=1e-7
    )
    scaler = torch.cuda.amp.GradScaler(enabled=MIXED_PRECISION and device.type == "cuda")
    early_stopper = EarlyStopper(patience=EARLY_STOP_PATIENCE, min_delta=MIN_DELTA, mode="max")

    trial_dir = CHECKPOINT_DIR / f"trial_{trial_index:02d}_lr_{learning_rate:g}"
    trial_dir.mkdir(parents=True, exist_ok=True)
    best_checkpoint_path = trial_dir / "best.pt"

    best = {
        "trial_index": trial_index,
        "learning_rate": learning_rate,
        "best_epoch": None,
        "best_val_dice": -math.inf,
        "best_val_iou": None,
        "best_val_loss": None,
        "checkpoint_path": str(best_checkpoint_path),
        "stopped_epoch": None,
        "stop_reason": "max_epochs",
    }
    history = []

    print(f"\n=== Trial {trial_index}/3 | learning_rate={learning_rate:g} ===")
    for epoch in range(1, MAX_EPOCHS + 1):
        train_metrics = run_epoch(model, train_loader, criterion, optimizer, scaler, epoch, phase="train")
        val_metrics = run_epoch(model, val_loader, criterion, optimizer=None, scaler=None, epoch=epoch, phase="val")
        scheduler.step(val_metrics["loss"])

        record = {
            "trial_index": trial_index,
            "learning_rate": learning_rate,
            "epoch": epoch,
            **{f"train_{k}": v for k, v in train_metrics.items()},
            **{f"val_{k}": v for k, v in val_metrics.items()},
            "current_lr": optimizer.param_groups[0]["lr"],
        }
        history.append(record)

        should_stop, improved = early_stopper.step(val_metrics["dice"])
        if improved:
            best.update({
                "best_epoch": epoch,
                "best_val_dice": val_metrics["dice"],
                "best_val_iou": val_metrics["iou"],
                "best_val_loss": val_metrics["loss"],
            })
            save_checkpoint(best_checkpoint_path, model, optimizer, epoch, learning_rate, val_metrics, trial_index)
            print(
                f"Trial {trial_index} epoch {epoch}: new best val Dice={val_metrics['dice']:.4f}, "
                f"IoU={val_metrics['iou']:.4f}, loss={val_metrics['loss']:.4f}"
            )
        else:
            print(
                f"Trial {trial_index} epoch {epoch}: val Dice={val_metrics['dice']:.4f}, "
                f"best={best['best_val_dice']:.4f}, bad_epochs={early_stopper.bad_epochs}/{EARLY_STOP_PATIENCE}"
            )

        pd.DataFrame(history).to_csv(trial_dir / "history.csv", index=False)
        if should_stop:
            best["stopped_epoch"] = epoch
            best["stop_reason"] = f"early_stop_patience_{EARLY_STOP_PATIENCE}"
            print(f"Stopping trial {trial_index}: no validation Dice improvement for {EARLY_STOP_PATIENCE} epochs.")
            break

    if best["stopped_epoch"] is None:
        best["stopped_epoch"] = history[-1]["epoch"]
    return best, history

# Model Training Pipeline

This is the long-running cell. It runs exactly three learning-rate trials from `LEARNING_RATE_TRIALS`. Each trial trains SegFormer MiT-B2 for up to 50 epochs and stops early when validation Dice does not improve for 3 consecutive epochs. The best checkpoint for each trial is the epoch with the highest validation Dice.

In [ ]:
all_history = []
trial_summaries = []

for trial_index, lr in enumerate(LEARNING_RATE_TRIALS, start=1):
    best, history = train_trial(trial_index, lr)
    trial_summaries.append(best)
    all_history.extend(history)

history_df = pd.DataFrame(all_history)
summary_df = pd.DataFrame(trial_summaries).sort_values("best_val_dice", ascending=False).reset_index(drop=True)

history_path = OUTPUT_DIR / "trial_history.csv"
summary_path = OUTPUT_DIR / "trial_summary.csv"
history_df.to_csv(history_path, index=False)
summary_df.to_csv(summary_path, index=False)

best_trial = summary_df.iloc[0].to_dict()
best_summary_path = OUTPUT_DIR / "best_trial_summary.json"
best_summary_path.write_text(json.dumps(best_trial, indent=2))

print("Trial summary:")
display(summary_df)
print(f"Best LR: {best_trial['learning_rate']} | best epoch: {int(best_trial['best_epoch'])} | val Dice: {best_trial['best_val_dice']:.4f}")
print(f"Wrote {history_path}")
print(f"Wrote {summary_path}")
print(f"Wrote {best_summary_path}")

# Evaluation on the Held-Out Test Split

Load the best validation checkpoint across all learning-rate trials, then evaluate it on the 150-image test split.

In [ ]:
def load_best_model(checkpoint_path: str):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model = SegformerForgeryDetector(
        encoder_name=checkpoint.get("encoder_name", ENCODER_NAME),
        encoder_weights=None,
    ).to(device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()
    return model, checkpoint


@torch.no_grad()
def collect_predictions(model, loader, criterion, phase="eval"):
    model.eval()
    probs_batches = []
    target_batches = []
    loss_meter = AverageMeter()
    bce_meter = AverageMeter()
    overlap_meter = AverageMeter()

    for batch in tqdm(loader, desc=f"Collecting {phase} predictions", leave=False):
        images = batch["image"].to(device, non_blocking=True)
        masks = batch["mask"].to(device, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=MIXED_PRECISION and device.type == "cuda"):
            logits = model(images)
            loss, bce_loss, overlap_loss = criterion(logits, masks)
        batch_size = images.size(0)
        loss_meter.update(loss.item(), batch_size)
        bce_meter.update(bce_loss.item(), batch_size)
        overlap_meter.update(overlap_loss.item(), batch_size)
        probs_batches.append(torch.sigmoid(logits).detach().cpu().to(torch.float16))
        target_batches.append(masks.detach().cpu().to(torch.uint8))

    return {
        "probs": probs_batches,
        "targets": target_batches,
        "loss": loss_meter.avg,
        "bce_loss": bce_meter.avg,
        "overlap_loss": overlap_meter.avg,
    }


def metrics_from_cached_predictions(cached, threshold: float, smooth=1.0):
    dice_values = []
    iou_values = []
    acc_values = []
    for probs, targets in zip(cached["probs"], cached["targets"]):
        probs = probs.float()
        targets = targets.float()
        preds = (probs > threshold).float()
        dims = (1, 2, 3)
        intersection = (preds * targets).sum(dim=dims)
        pred_sum = preds.sum(dim=dims)
        target_sum = targets.sum(dim=dims)
        union = pred_sum + target_sum - intersection
        dice = (2.0 * intersection + smooth) / (pred_sum + target_sum + smooth)
        iou = (intersection + smooth) / (union + smooth)
        acc = (preds == targets).float().mean(dim=dims)
        dice_values.extend(dice.tolist())
        iou_values.extend(iou.tolist())
        acc_values.extend(acc.tolist())

    return {
        "loss": float(cached["loss"]),
        "bce_loss": float(cached["bce_loss"]),
        "overlap_loss": float(cached["overlap_loss"]),
        "dice": float(np.mean(dice_values)),
        "iou": float(np.mean(iou_values)),
        "pixel_acc": float(np.mean(acc_values)),
        "threshold": float(threshold),
    }


best_model, best_checkpoint = load_best_model(best_trial["checkpoint_path"])
criterion = CombinedLoss(pos_weight=POS_WEIGHT_TENSOR)

val_cached = collect_predictions(best_model, val_loader, criterion, phase="validation")
threshold_records = [metrics_from_cached_predictions(val_cached, threshold) for threshold in THRESHOLD_SWEEP]
threshold_df = pd.DataFrame(threshold_records).sort_values("dice", ascending=False).reset_index(drop=True)
threshold_path = OUTPUT_DIR / "validation_threshold_sweep.csv"
threshold_df.to_csv(threshold_path, index=False)
best_threshold = float(threshold_df.iloc[0]["threshold"])

print("Validation threshold sweep top rows:")
display(threshold_df.head(10))
print(f"Selected threshold={best_threshold:.2f} based on validation Dice")

# Evaluate the chosen threshold on the held-out test split exactly once.
test_cached = collect_predictions(best_model, test_loader, criterion, phase="test")
test_metrics = metrics_from_cached_predictions(test_cached, best_threshold)

test_result = {
    "best_trial": best_trial,
    "selected_threshold": best_threshold,
    "validation_threshold_sweep_csv": str(threshold_path),
    "test_metrics": test_metrics,
    "test_count": len(test_df),
    "split_csv": str(split_csv),
}
test_metrics_path = OUTPUT_DIR / "test_metrics.json"
test_metrics_path.write_text(json.dumps(test_result, indent=2))

print("Test metrics from best validation checkpoint and selected threshold:")
print(json.dumps(test_metrics, indent=2))
print(f"Wrote {threshold_path}")
print(f"Wrote {test_metrics_path}")

# Visualize Predictions

Show qualitative test results using the best validation checkpoint.

In [ ]:
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406])
IMAGENET_STD = np.array([0.229, 0.224, 0.225])


def denormalize_image(tensor: torch.Tensor) -> np.ndarray:
    image = tensor.detach().cpu().permute(1, 2, 0).numpy()
    image = np.clip((image * IMAGENET_STD + IMAGENET_MEAN) * 255.0, 0, 255).astype(np.uint8)
    return image


def visualize_predictions(model, dataset, count=6, threshold=THRESHOLD, seed=SEED):
    rng = random.Random(seed)
    indices = rng.sample(range(len(dataset)), min(count, len(dataset)))
    fig, axes = plt.subplots(len(indices), 4, figsize=(16, 4 * len(indices)))
    if len(indices) == 1:
        axes = np.expand_dims(axes, axis=0)

    model.eval()
    for row_idx, ds_idx in enumerate(indices):
        sample = dataset[ds_idx]
        image_tensor = sample["image"]
        mask_tensor = sample["mask"]
        with torch.no_grad():
            logits = model(image_tensor.unsqueeze(0).to(device))
            prob = torch.sigmoid(logits)[0, 0].cpu().numpy()
        pred = (prob > threshold).astype(np.uint8)
        image = denormalize_image(image_tensor)
        mask = mask_tensor[0].cpu().numpy()

        overlay = image.copy()
        overlay[pred > 0] = (0.55 * overlay[pred > 0] + 0.45 * np.array([255, 0, 0])).astype(np.uint8)

        axes[row_idx, 0].imshow(image)
        axes[row_idx, 0].set_title(f"Image\n{sample['tamper_id']}")
        axes[row_idx, 1].imshow(mask, cmap="gray")
        axes[row_idx, 1].set_title("Ground truth mask")
        axes[row_idx, 2].imshow(prob, cmap="magma", vmin=0, vmax=1)
        axes[row_idx, 2].set_title("Predicted probability")
        axes[row_idx, 3].imshow(overlay)
        axes[row_idx, 3].set_title("Prediction overlay")
        for col in range(4):
            axes[row_idx, col].axis("off")

    fig.tight_layout()
    return fig


test_dataset_for_viz = TamperSegmentationDataset(test_df, eval_transform)
fig = visualize_predictions(best_model, test_dataset_for_viz, count=6, threshold=best_threshold)
viz_path = OUTPUT_DIR / "best_model_test_predictions.png"
fig.savefig(viz_path, dpi=160, bbox_inches="tight")
plt.show()
print(f"Wrote {viz_path}")

# Optional: Upload Outputs Back to GCS

Set `UPLOAD_OUTPUTS_TO_GCS = True` in the config cell and rerun this cell if you want the split IDs, histories, summaries, checkpoints, and visualizations uploaded to:

`gs://maverick91/audit_finding_project/screenshot_tamper_final/segformer_results/<RUN_NAME>/`

In [ ]:
if UPLOAD_OUTPUTS_TO_GCS:
    upload_outputs_to_gcs(OUTPUT_DIR, OUTPUT_GCS_PREFIX)
else:
    print("UPLOAD_OUTPUTS_TO_GCS is False; outputs remain local.")
    print(f"Local outputs: {OUTPUT_DIR}")

# Training Completed - Final Result Object

This cell prints the key outputs to report: best learning rate, best epoch, best validation Dice/IoU/loss, held-out test metrics, and the files containing recorded IDs and trial history.

In [ ]:
final_result = {
    "run_name": RUN_NAME,
    "sample_size": SAMPLE_SIZE,
    "split_counts": sample_df["split"].value_counts().to_dict(),
    "learning_rate_trials": LEARNING_RATE_TRIALS,
    "max_epochs": MAX_EPOCHS,
    "early_stop_patience": EARLY_STOP_PATIENCE,
    "best_trial": best_trial,
    "selected_threshold": best_threshold,
    "test_metrics": test_metrics,
    "output_dir": str(OUTPUT_DIR),
    "split_ids_csv": str(split_csv),
    "trial_history_csv": str(history_path),
    "trial_summary_csv": str(summary_path),
    "validation_threshold_sweep_csv": str(threshold_path),
    "test_metrics_json": str(test_metrics_path),
}
print(json.dumps(final_result, indent=2))